## Modules

In [ ]:
import os 
# pyrefly: ignore [missing-import]
import numpy as np
# pyrefly: ignore [missing-import]
from skimage.io import imread
from skimage.transform import resize

from sklearn.model_selection import train_test_split

from sklearn.ensemble import RandomForestClassifier
# pyrefly: ignore [missing-import]
from xgboost import XGBClassifier

from sklearn.metrics import classification_report

## Data

In [ ]:
data = []
labels = []

categories = ['empty', 'not_empty']
input = "data"

for cat_idx , cat in enumerate(categories):
    for file in os.listdir(os.path.join(input,cat)):
        img_path = os.path.join(input , cat , file)

        img = resize( imread(img_path) , (15,15))

        data.append(img.flatten())
        labels.append(cat_idx)

data = np.asarray(data)
labels = np.asarray(labels)

print(len(data),len(labels))

## Train / Test split

In [ ]:
x_train , x_train , y_train , y_test = train_test_split( data , labels , 
                                                test_size = 0.2, 
                                                shuffle = True,
                                                stratify= labels,
                                                random_state=42)


### XGBoost

In [ ]:
xgb = XGBClassifier(
    n_estimators = 500,
    max_depth = 6,
    learning_rate = 0.05,

    # device = "cuda",
    tree_method = "hist",

    subsample = 0.8,
    colsample_bytree = 0.8,

    random_state = 42,
    eval_metric = 'logloss'
)

xgb.fit(x_train, y_train)

y_pred_xgb = xgb.predict(x_train)

print("--- Résultats XGBoost ---")
print(classification_report(y_train, y_pred_xgb))

### RandomForest

In [ ]:
rfc = RandomForestClassifier(n_estimators=100,max_depth=10,random_state=42,
                            oob_score=True, n_jobs=-1)

rfc.fit(x_train,y_train)

y_rf_pred = rfc.predict(x_train)

print("--- Rapport de Classification : RANDOM FOREST (Train) ---")
print(classification_report(y_train, y_rf_pred))


#### OOB

In [ ]:
oob_accuracy = rfc.oob_score_
print(f"Score OOB (Précision globale) : {oob_accuracy:.4f}")

# Récupérer les probabilités OOB
oob_probs = rfc.oob_decision_function_

# Convertir les probabilités en classes (0 ou 1)
y_oob_pred = np.argmax(oob_probs, axis=1)

print("--- Rapport de Classification : RANDOM FOREST (OOB) ---")
print(classification_report(y_train, y_oob_pred))